# Average Metric Repair — Experiments

Tidy, reproducible, parallel experiment harness.

**Design** (one principle: *measure per instance, store long, aggregate later*):
- `run_instance(...)` runs every algorithm on the **same** graph instance, times each one
  (`perf_counter`), checks the cover with `verifier`/`iomr_verifier`, and returns **one row per
  (instance, component, algorithm)** — never pre-averaged.
- Parallelism is **across instances** (a fork `multiprocessing.Pool`), with each worker kept
  single-threaded — so per-algorithm runtimes stay comparable while the sweep runs N-way faster.
- Every saved file carries the **git commit** and a timestamp (provenance), and lands in `results/`
  (gitignored) as gzipped CSV.

Aggregate at analysis time: `summarize(df)` gives means/std/runtime per (algorithm, n, p).

## Setup

In [ ]:
import os
# Keep each parallel worker single-threaded so tasks don't oversubscribe the 10 cores.
# (For these to bind the BLAS backend they must be set BEFORE numpy imports below, i.e. before
#  the %run on a fresh kernel. Restart the kernel if you change them.)
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

%run Packages_and_Functions.ipynb

import pandas as pd
import numpy as np
import time, subprocess, datetime
import multiprocessing as mp
from time import perf_counter

## Registries

`ALGORITHMS` is the list of (name, variant, model, callable) the harness times on each component.
`variant` is `on_G` (run on the component) vs `complete` (run on its metric completion); `model`
picks the validity checker (`verifier` for general MR, `iomr_verifier` for increase-only).

Old `algo_dict` index → new (algorithm, variant): 0 domr · 1/2 pivot on_G/complete ·
3/4 left_edge on_G/complete · 5/6 spc_general on_G/complete · 7/8 spc_iomr on_G/complete.

In [ ]:
GENERATORS = {
    "geometric":   random_geometric_weighted_graph,
    "uniform":     random_uniform_weighted_graph,
    "exponential": random_exponential_weighted_graph,
}

# (name, variant, model, fn(CC, comCC))
ALGORITHMS = [
    ("domr",        "on_G",     "general", lambda CC, comCC: domr_alg(CC)),
    ("pivot",       "on_G",     "general", lambda CC, comCC: pivot_heuristic(CC)),
    ("pivot",       "complete", "general", lambda CC, comCC: pivot_heuristic(comCC)),
    ("left_edge",   "on_G",     "iomr",    lambda CC, comCC: left_edge_heuristic(CC)),
    ("left_edge",   "complete", "iomr",    lambda CC, comCC: left_edge_heuristic(comCC)),
    ("spc_general", "on_G",     "general", lambda CC, comCC: shortest_path_cover(CC)),
    ("spc_general", "complete", "general", lambda CC, comCC: shortest_path_cover(comCC)),
    ("spc_iomr",    "on_G",     "iomr",    lambda CC, comCC: shortest_path_cover(CC, general=0)),
    ("spc_iomr",    "complete", "iomr",    lambda CC, comCC: shortest_path_cover(comCC, general=0)),
]

## Core: run one instance

Returns tidy rows. Each algorithm is timed in isolation (sequential within the instance), so the
`runtime_s` column is comparable across algorithms even when many instances run in parallel.
`compute_valid=False` skips the verifier calls for a faster sweep.

In [ ]:
def run_instance(generator, n, p, seed, trial, compute_valid=True):
    """Run every algorithm on one freshly generated graph; return a list of tidy row dicts."""
    seed_all(seed)                                   # reproducible: the seed is recorded in every row
    G = GENERATORS[generator](int(n), float(p))
    rows = []
    for ci, CC in enumerate(G.connected_components_subgraphs()):
        comCC = complete(CC)
        n_broken = len(domr_alg(CC))                 # broken edges in this component (DOMR size)
        for name, variant, model, fn in ALGORITHMS:
            t0 = perf_counter()
            S = fn(CC, comCC)
            dt = perf_counter() - t0
            valid = -1
            if compute_valid:
                tgt = CC if variant == "on_G" else comCC
                valid = int(iomr_verifier(tgt, S) if model == "iomr" else verifier(tgt, S))
            rows.append(dict(
                generator=generator, n=int(n), p=float(p), seed=int(seed), trial=int(trial),
                component=ci, algorithm=name, variant=variant, model=model,
                cover_size=len(S), runtime_s=dt, valid=valid,
                comp_n=CC.num_verts(), comp_edges=CC.num_edges(), n_broken=n_broken))
    return rows

def _worker(arg):                                    # module-level so the fork Pool can pickle it
    task, compute_valid = arg
    return run_instance(*task, compute_valid=compute_valid)

## Sweep, provenance, save

In [ ]:
def build_tasks(generators, ns, ps=None, p_of_n=None, trials=1, base_seed=0):
    """Cartesian sweep generators x ns x (ps | p_of_n(n)) x trials -> list of task tuples.
    Pass ps (a list of p values, e.g. a p-sweep at fixed n) OR p_of_n (a function n -> p)."""
    tasks = []
    for gen in generators:
        for n in ns:
            pvals = [float(p_of_n(n))] if p_of_n is not None else [float(x) for x in ps]
            for p in pvals:
                for trial in range(int(trials)):
                    tasks.append((gen, int(n), float(p), int(base_seed) + len(tasks), int(trial)))
    return tasks

def run_sweep(tasks, parallel=True, n_jobs=None, compute_valid=True):
    """Run all tasks and return one tidy DataFrame. Parallel across instances via a fork Pool;
    set parallel=False if forking is unstable in your kernel (results are identical either way)."""
    work = [(t, compute_valid) for t in tasks]
    if parallel:
        with mp.get_context("fork").Pool(n_jobs or mp.cpu_count()) as pool:
            res = pool.map(_worker, work)
    else:
        res = [_worker(w) for w in work]
    return pd.DataFrame([r for sub in res for r in sub])

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return "unknown"

def save_results(df, name, outdir="results"):
    """Write the tidy rows to results/<name>_<timestamp>.csv.gz, stamped with the git commit."""
    os.makedirs(outdir, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    out = df.copy()
    out["code_version"] = git_sha()
    out["run_timestamp"] = stamp
    path = os.path.join(outdir, f"{name}_{stamp}.csv.gz")
    out.to_csv(path, index=False)
    print(f"saved {len(out)} rows -> {path}  (code {git_sha()})")
    return path

## Analysis helpers

`per_instance` sums each algorithm's cover/runtime over a graph's components (the per-G total, the
quantity the paper compares); `summarize` then aggregates over trials. Everything else (CIs,
medians, per-component views, filtering to `n_broken > 0`) is a one-liner on the raw frame.

In [ ]:
def per_instance(df):
    """Collapse components: one row per (instance, algorithm) with the per-graph total cover/runtime."""
    keys = ["generator", "n", "p", "seed", "trial", "algorithm", "variant", "model"]
    return df.groupby(keys, as_index=False).agg(
        cover_size=("cover_size", "sum"),
        runtime_s=("runtime_s", "sum"),
        valid=("valid", "min"))

def summarize(df, by=("generator", "algorithm", "variant", "n", "p")):
    """Mean/std cover size, mean runtime (ms), and validity rate over trials."""
    inst = per_instance(df)
    g = inst.groupby(list(by))
    out = g.agg(
        cover_mean=("cover_size", "mean"),
        cover_std=("cover_size", "std"),
        runtime_ms=("runtime_s", lambda s: 1000.0 * s.mean()),
        valid_rate=("valid", "mean"),
        reps=("cover_size", "size"))
    return out.round(int(3))

## Example sweeps (sanity scale)

Small params so these run quickly. Scale up by enlarging `ns` / `trials`. Uncomment `save_results`
to persist. The two `build_tasks` calls replace the old `experiment_heuristics_on_G_and_KG`
(sweep n) and `experiment_algs_as_function_of_p` (sweep p).

In [ ]:
# --- sweep over n at p = 0.3 ---
tasks_n = build_tasks(["geometric"], ns=[40, 60, 80], p_of_n=lambda n: 0.3, trials=5, base_seed=0)
df_n = run_sweep(tasks_n, parallel=True, n_jobs=8)   # parallel=False if your kernel dislikes fork
print(f"{len(df_n)} rows | all covers valid: {bool((df_n['valid'] == 1).all())}")
# save_results(df_n, "alg_sweep_n")
summarize(df_n)

In [ ]:
# --- sweep over p at n = 80 ---
tasks_p = build_tasks(["geometric"], ns=[80], ps=list(np.linspace(0.1, 0.9, 5)), trials=5, base_seed=1000)
df_p = run_sweep(tasks_p, parallel=True, n_jobs=8)
print(f"{len(df_p)} rows | all covers valid: {bool((df_p['valid'] == 1).all())}")
# save_results(df_p, "alg_sweep_p")
summarize(df_p)

## Threshold experiments (broken-cycle probability)

A separate family (different metric: how often a random graph has a broken induced cycle, vs.
`p = N^{-q}`). Already seeded; not yet converted to the tidy/parallel harness above — a natural
next step if you want error bars and parallelism here too.

In [ ]:
### Experiment - Demonstrate Thereshold

def experiment_demonstrate_threshold(N = 30,AVG = 30, ps=20, seed=0):
    seed_all(seed)
    delta = 1.9/ps
    P = [2 - i*delta for i in range(ps)]
    res = np.zeros((2,ps))
    for j,q in enumerate(P):
        print(f"finished {100*j//ps}%")
        p = 1/np.power(N,q)
        tot = 0
        for trial in range(AVG):
            G = random_geometric_weighted_graph(N,p)
            while G.size() == 0: # Make sure the graph has edges
                G = random_geometric_weighted_graph(N,p)
            w = get_weights_vector(G,make_index_encoding(G))
            phi,count = induced_cycle_matrix(G)
            V = phi@w # potential deficit values for cycles
            tot +=np.count_nonzero(V < 0)
        res[0,j] = tot/AVG ### Broken Cycles
        res[1,j] = count # how many induced cycles 
    return res


### Experiment - Demonstrate Thereshold

def experiment_demonstrate_threshold_uniform(N = 30,AVG = 30, ps=20, seed=0):
    seed_all(seed)
    delta = 1.9/ps
    P = [2 - i*delta for i in range(ps)]
    res = np.zeros((2,ps))
    for j,q in enumerate(P):
        print(f"finished {100*j//ps}%")
        p = 1/np.power(N,q)
        tot = 0
        for trial in range(AVG):
            G = random_uniform_weighted_graph(N,p)
            while G.size() == 0: # Make sure the graph has edges
                G = random_uniform_weighted_graph(N,p)
            w = get_weights_vector(G,make_index_encoding(G))
            phi,count = induced_cycle_matrix(G)
            V = phi@w # potential deficit values for cycles
            tot +=np.count_nonzero(V < 0)
        res[0,j] = tot/AVG ### Broken induced Cycles
        res[1,j] = count # how many induced cycles 
    return res

def experiment_demonstrate_threshold_prob(N = 30,AVG = 30, ps=50,std=False, seed=0):
    seed_all(seed)
    delta = 1.9/ps
    P = [2 - i*delta for i in range(ps)]
    res = np.zeros((2,ps))
    res_std = np.zeros((2,ps))
    for j,q in enumerate(P):
        print(f"finished {100*j//ps}%")
        p = 1/np.power(N,q)
        samples_broken=np.zeros(AVG)
        samples_cycles=np.zeros(AVG)
        has_broken = 0
        has_cycle = 0
        for trial in range(AVG):
            G = random_geometric_weighted_graph(N,p)
            while G.size() == 0: # Make sure the graph has edges
                G = random_geometric_weighted_graph(N,p)
            w = get_weights_vector(G,make_index_encoding(G))
            phi,count = induced_cycle_matrix(G)
            V = phi@w # potential deficit values for cycles
            if count > 0:
                samples_cycles[trial] = 1
                has_cycle +=1
            if np.count_nonzero(V < 0) > 0:
                samples_broken[trial] = 1
                has_broken +=1
        res[0,j] = has_broken/AVG ### Broken Cycles
        res[1,j] = has_cycle/AVG # how many induced cycles 
        res_std[0,j]=np.std(samples_broken)/np.sqrt(AVG)
        res_std[1,j]=np.std(samples_cycles)/np.sqrt(AVG)         
    if std:
        return res,res_std
    else:
        return res